In [1]:
!pip install joblib tqdm scikit-learn folktables

In [2]:
"""
AGAD: Auditor-Guided Adversarial Debiasing
==========================================
Fixes applied vs. original (all bugs from review doc addressed):

CRITICAL FIXES
1.  ACS scaler leakage       - scaler now fit on train only, transform test separately
2.  Cache invalidation        - cache key includes a VERSION hash; change code → new cache
3.  Auditor overfit           - worst subgroup validated on held-out val set before adoption;
                                also adopted subgroup is added to val/test evaluation
4.  GRL double-scaling bug    - grl_alpha appears ONCE (inside GRL); loss just adds grl_loss
5.  Adversary layer reset     - set_n_attrs keeps old hidden weights, only grows output layer
6.  Real adversary steps      - adv_steps / adv_lr_mult are now actually used; two-phase
                                per-batch: (a) adversary update, (b) encoder+task update
7.  Fairness penalty mismatch - EO penalty uses hard thresholded predictions (detached),
                                matching evaluation; soft version kept as secondary term
8.  Intersectionality fix     - make_intersectional now returns TRUE intersection group
                                (both attrs = minority), not majority-vs-rest
9.  Early stopping            - best model tracks (fairness, AUC) jointly; pure fairness
                                only selected when AUC is within 1% of best-seen AUC
10. Auditor val/test update   - discovered subgroup propagated to sv_val_list & sv_te_list

LOGICAL FIXES
11. find_hardest_attr mode    - mode parameter now actually used (eo vs dp)
12. reweights use all attrs   - compute_label_reweights now averages weights across all attrs
13. oversampling uses all     - balanced loader uses intersectional/combined group
14. Unused params removed     - beta, mode dead param cleaned up; adv_steps/adv_lr_mult used

METRIC ADDITIONS
15. Demographic Parity (DP)   - reported separately; DP training objective optional
16. Calibration / ECE         - expected calibration error reported
17. Multi-seed CI support     - run N_SEEDS seeds, report mean ± 95% CI via bootstrap
18. per_attr reported cleanly

NOVELTY NOTE
-----------
The AGAD loop (auditor discovers worst subgroup → adds it as new adversarial target → 
retrain → repeat) is inspired by Kearns et al. (ICML 2018) "Preventing Fairness 
Gerrymandering" and Zhang et al. (AIES 2018) adversarial debiasing, but their combination 
in an online training loop with GRL is novel. Key differences from prior work:
- Kearns et al. use a game-theoretic oracle (not neural adversary) for auditing
- Zhang et al. do not do iterative subgroup discovery during training
- Our loop is closest to "Fair Game" (2025) but uses EO not DP and targets gerrymandering
"""

FAST_MODE = False      # set True for quick smoke-test
N_SEEDS   = 3          # seeds for confidence intervals (set 1 for single run)

import os, gc, warnings, json, time, copy, hashlib
os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    from folktables import ACSDataSource, ACSEmployment, ACSPublicCoverage
    HAS_FOLKTABLES = True
except ImportError:
    HAS_FOLKTABLES = False
    print("[warn] folktables not installed. Run: pip install folktables")

_KAGGLE_BASE = "/kaggle/input/datasets/lateglue/all-datasets"
if not os.path.isdir(_KAGGLE_BASE):
    _KAGGLE_BASE = "data"

DATA_PATHS = {
    "adult":  f"{_KAGGLE_BASE}/adult.csv",
    "compas": f"{_KAGGLE_BASE}/compas-scores-two-years.csv",
}

RUN_DATASETS = ["adult", "compas", "acs_employment", "acs_public_coverage"]

SEED        = 42
SEEDS       = list(range(SEED, SEED + N_SEEDS))
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_EO   = 0.05
TARGET_DP   = 0.10   # demographic parity threshold (looser by convention)
AUDIT_EVERY = 5
CACHE_VERSION = "v3"   # bump this if you change preprocessing

DATASET_CONFIG = {
    "adult":               {"sens_attrs": ["sens_sex", "sens_race"], "reweight_labels": True},
    "compas":              {"sens_attrs": ["sens_race", "sens_sex"], "reweight_labels": True},
    "acs_employment":      {"sens_attrs": ["sens_race", "sens_sex"], "reweight_labels": True},
    "acs_public_coverage": {"sens_attrs": ["sens_race", "sens_sex"], "reweight_labels": True},
}

_FAST_OVERRIDES = {
    "adult":               {"epochs": 60,  "pre_epochs": 20, "patience": 10},
    "compas":              {"epochs": 60,  "pre_epochs": 10, "patience": 10},
    "acs_employment":      {"epochs": 40,  "pre_epochs": 15, "patience": 8},
    "acs_public_coverage": {"epochs": 40,  "pre_epochs": 15, "patience": 8},
}

BASE_PARAMS = {
    "adult": {
        "hidden_dim": 128, "dropout": 0.15, "lr": 4e-4, "batch": 512,
        "epochs": 150, "fairness_coef": 8.0, "warmup": 40,
        "eval_every": 5, "patience": 30, "min_acc_floor": 0.78,
        # FIXED: adv_lr_mult and adv_steps are now USED
        "adv_lr_mult": 5.0, "adv_steps": 3, "pre_epochs": 60,
        "min_pos_frac": 0.12, "max_pos_frac": 0.35,
        "task_lw": 5.0, "mixup_alpha": 0.0, "grl_max": 0.3,
    },
    "compas": {
        "hidden_dim": 128, "dropout": 0.15, "lr": 4e-4, "batch": 256,
        "epochs": 200, "fairness_coef": 20.0, "warmup": 15,
        "eval_every": 5, "patience": 40, "min_acc_floor": 0.58,
        "adv_lr_mult": 5.0, "adv_steps": 3, "pre_epochs": 30,
        "min_pos_frac": 0.30, "max_pos_frac": 0.75,
        "task_lw": 2.0, "mixup_alpha": 0.0, "grl_max": 0.8,
    },
    "acs_employment": {
        "hidden_dim": 128, "dropout": 0.15, "lr": 4e-4, "batch": 512,
        "epochs": 120, "fairness_coef": 10.0, "warmup": 30,
        "eval_every": 5, "patience": 25, "min_acc_floor": 0.75,
        "adv_lr_mult": 5.0, "adv_steps": 3, "pre_epochs": 40,
        "min_pos_frac": 0.55, "max_pos_frac": 0.85,
        "task_lw": 4.0, "mixup_alpha": 0.0, "grl_max": 0.4,
    },
    "acs_public_coverage": {
        "hidden_dim": 128, "dropout": 0.15, "lr": 4e-4, "batch": 512,
        "epochs": 120, "fairness_coef": 10.0, "warmup": 30,
        "eval_every": 5, "patience": 25, "min_acc_floor": 0.72,
        "adv_lr_mult": 5.0, "adv_steps": 3, "pre_epochs": 40,
        "min_pos_frac": 0.25, "max_pos_frac": 0.60,
        "task_lw": 4.0, "mixup_alpha": 0.0, "grl_max": 0.4,
    },
}

def _get_params(ds_key):
    p = dict(BASE_PARAMS[ds_key])
    if FAST_MODE:
        p.update(_FAST_OVERRIDES[ds_key])
    return p

METHODS = ["lr_baseline", "agad_no_auditor", "agad"]


# ─── Utilities ────────────────────────────────────────────────────────────────

def set_seed(s=SEED):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    np.random.seed(s)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.array(X)

def ensure_binary(sv):
    sv = np.array(sv).flatten()
    u = np.unique(sv)
    if len(u) == 1:
        return np.zeros(len(sv), dtype=int)
    if len(u) == 2:
        return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def clean_numeric(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.fillna(s.median() if s.notna().any() else 0)

def stratified_split(X, y, sens_list, test_size=0.2):
    key = np.array(y).astype(str).copy()
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    small = u[c < 2]
    key[np.isin(key, small)] = np.array(y)[np.isin(key, small)].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=SEED)
    except Exception:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=SEED)

def build_val_split(y_train, frac=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = rng.choice(len(y_train), size=max(1, int(len(y_train) * frac)), replace=False)
    mask = np.ones(len(y_train), dtype=bool)
    mask[val_idx] = False
    return np.where(mask)[0], val_idx

def safe_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return float("nan")
        return roc_auc_score(y_true, y_score)
    except Exception:
        return float("nan")

def bootstrap_ci(values, n_boot=2000, ci=0.95, seed=0):
    """Return (mean, lower, upper) bootstrap CI."""
    rng = np.random.RandomState(seed)
    arr = np.array(values)
    boots = [rng.choice(arr, len(arr), replace=True).mean() for _ in range(n_boot)]
    lo = np.percentile(boots, 100 * (1 - ci) / 2)
    hi = np.percentile(boots, 100 * (1 - (1 - ci) / 2))
    return float(arr.mean()), float(lo), float(hi)

# ─── FIXED: True intersectional group (minority × minority), not majority-vs-rest ──

def make_intersectional(sv_list):
    """
    Returns binary array: 1 if ALL attributes are in the MINORITY class.
    This is a proper intersectional group (e.g. non-white AND female).
    """
    if len(sv_list) == 1:
        return sv_list[0].copy()
    # minority of each attribute = value with fewer samples
    result = np.ones(len(sv_list[0]), dtype=int)
    for sv in sv_list:
        sv_b = ensure_binary(sv)
        # minority = whichever of 0/1 has fewer samples
        n0, n1 = (sv_b == 0).sum(), (sv_b == 1).sum()
        minority_val = 0 if n0 < n1 else 1
        result = result & (sv_b == minority_val)
    return result.astype(int)

def find_hardest_attr(y, sv_list, mode="eo"):
    """
    FIX: mode is now actually used.
    mode='eo'  → worst max(|ΔTPR|, |ΔFPR|)
    mode='dp'  → worst |Δpositive_rate|
    """
    worst_idx = 0
    worst_val = -1.0
    for i, sv in enumerate(sv_list):
        sv_b = ensure_binary(sv)
        if len(np.unique(sv_b)) < 2:
            continue
        if mode == "dp":
            r0 = y[sv_b == 0].mean() if (sv_b == 0).sum() > 0 else 0.5
            r1 = y[sv_b == 1].mean() if (sv_b == 1).sum() > 0 else 0.5
            val = abs(r0 - r1)
        else:  # eo
            pos0 = (sv_b == 0) & (y == 1)
            pos1 = (sv_b == 1) & (y == 1)
            neg0 = (sv_b == 0) & (y == 0)
            neg1 = (sv_b == 1) & (y == 0)
            tpr0 = y[pos0].mean() if pos0.sum() > 0 else 0.5
            tpr1 = y[pos1].mean() if pos1.sum() > 0 else 0.5
            fpr0 = (1 - y)[neg0].mean() if neg0.sum() > 0 else 0.5
            fpr1 = (1 - y)[neg1].mean() if neg1.sum() > 0 else 0.5
            val = max(abs(tpr0 - tpr1), abs(fpr0 - fpr1))
        if val > worst_val:
            worst_val = val
            worst_idx = i
    return worst_idx

# ─── FIXED: reweights averaged over ALL attrs, not just hardest ─────────────

def compute_label_reweights(y, sv_list):
    """FIX #12: average reweights across all sensitive attributes."""
    y = np.array(y)
    n = len(y)
    p_y1 = np.clip(y.mean(), 1e-3, 1 - 1e-3)
    p_y0 = 1.0 - p_y1
    w_sum = np.zeros(n, dtype=np.float64)
    count = 0
    for sv in sv_list:
        sv_b = ensure_binary(sv)
        if len(np.unique(sv_b)) < 2:
            continue
        w = np.ones(n, dtype=np.float64)
        for g in [0, 1]:
            mask = sv_b == g
            if mask.sum() == 0:
                continue
            p_y1_g = np.clip(y[mask].mean(), 1e-3, 1 - 1e-3)
            p_y0_g = 1.0 - p_y1_g
            w[mask & (y == 1)] = p_y1 / p_y1_g
            w[mask & (y == 0)] = p_y0 / p_y0_g
        w_sum += w
        count += 1
    if count == 0:
        return np.ones(n, dtype=np.float32)
    w_avg = w_sum / count
    return (w_avg / w_avg.mean()).astype(np.float32)


# ─── Fairness Metrics ─────────────────────────────────────────────────────────

def compute_eo(y_true, y_pred, sv):
    """Equalized odds gap for one binary sensitive attribute."""
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2:
        return 0.0
    tprs, fprs = [], []
    for g in [0, 1]:
        pos = (sv == g) & (y_true == 1)
        neg = (sv == g) & (y_true == 0)
        tprs.append(float(y_pred[pos].mean()) if pos.sum() > 0 else 0.0)
        fprs.append(float(y_pred[neg].mean()) if neg.sum() > 0 else 0.0)
    return max(abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))

def compute_dp(y_pred, sv):
    """Demographic parity gap for one binary sensitive attribute."""
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2:
        return 0.0
    r0 = y_pred[sv == 0].mean() if (sv == 0).sum() > 0 else 0.5
    r1 = y_pred[sv == 1].mean() if (sv == 1).sum() > 0 else 0.5
    return abs(r0 - r1)

def compute_eo_multi(y_true, y_pred, sv_list):
    per = [compute_eo(y_true, y_pred, sv) for sv in sv_list]
    return per, max(per) if per else 0.0

def compute_dp_multi(y_pred, sv_list):
    per = [compute_dp(y_pred, sv) for sv in sv_list]
    return per, max(per) if per else 0.0

def compute_ece(y_true, y_prob, n_bins=10):
    """Expected calibration error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += mask.sum() * abs(acc - conf)
    return ece / len(y_true)

def compute_eo_subgroups(y_true, y_pred, sv_list):
    """Search over all boolean combinations of sensitive attributes."""
    n_attrs = len(sv_list)
    n = len(y_true)
    min_size = max(10, int(0.01 * n))
    worst = 0.0
    worst_grp = None
    worst_membership = np.zeros(n, dtype=int)
    for mask in range(1, 2 ** n_attrs):
        active = [i for i in range(n_attrs) if mask & (1 << i)]
        for vals in range(2 ** len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req):
                mem &= (sv_list[ai] == rv)
            if mem.sum() < min_size:
                continue
            sg_p  = y_pred[mem];  sg_t  = y_true[mem]
            ot_p  = y_pred[~mem]; ot_t  = y_true[~mem]
            if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2:
                continue
            tpr_sg  = sg_p[sg_t == 1].mean()  if (sg_t == 1).sum() > 0 else 0.
            tpr_oth = ot_p[ot_t == 1].mean()  if (ot_t == 1).sum() > 0 else 0.
            fpr_sg  = sg_p[sg_t == 0].mean()  if (sg_t == 0).sum() > 0 else 0.
            fpr_oth = ot_p[ot_t == 0].mean()  if (ot_t == 0).sum() > 0 else 0.
            eo = max(abs(tpr_sg - tpr_oth), abs(fpr_sg - fpr_oth))
            if eo > worst:
                worst = eo
                worst_grp = {"active": active, "vals": req, "size": int(mem.sum())}
                worst_membership = mem.astype(int)
    return worst, worst_grp, worst_membership

def run_auditor(y_true, y_pred, sv_list):
    pred_bin = (y_pred > 0.5).astype(int)
    return compute_eo_subgroups(y_true, pred_bin, sv_list)


# ─── Neural Network Modules ───────────────────────────────────────────────────

class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None

class GRL(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)

class Encoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        h = hidden_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, h // 2), nn.BatchNorm1d(h // 2), nn.GELU(), nn.Dropout(dropout * 0.5))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)

class TaskHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim // 2), nn.GELU(),
            nn.Linear(rep_dim // 2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, rep):
        return self.net(rep).squeeze(1)

class SensAttrClassifier(nn.Module):
    """
    FIX #5: set_n_attrs keeps hidden-layer weights; only reinitialises output.
    FIX #4: GRL alpha set once; NOT multiplied again in loss.
    """
    def __init__(self, rep_dim, n_attrs, hidden_dim, dropout):
        super().__init__()
        self.grl     = GRL(alpha=1.0)
        self.hidden  = nn.Sequential(
            nn.Linear(rep_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout))
        self.out     = nn.Linear(hidden_dim // 2, n_attrs)
        nn.init.kaiming_normal_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def set_alpha(self, a):
        self.grl.alpha = a

    def set_n_attrs(self, n):
        """Grow output layer; preserve hidden weights."""
        old_out = self.out
        new_out = nn.Linear(old_out.in_features, n).to(next(self.parameters()).device)
        # copy existing output weights where they fit
        with torch.no_grad():
            k = min(old_out.out_features, n)
            new_out.weight[:k] = old_out.weight[:k]
            new_out.bias[:k]   = old_out.bias[:k]
            if n > k:
                nn.init.kaiming_normal_(new_out.weight[k:])
                nn.init.zeros_(new_out.bias[k:])
        self.out = new_out

    def forward(self, z):
        return self.out(self.hidden(self.grl(z)))

def mixup_batch(xb, yb, sv_list_b, alpha=0.3, w_b=None):
    if alpha <= 0.0:
        return xb, yb, sv_list_b, w_b
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(xb.shape[0], device=xb.device)
    return (lam * xb + (1 - lam) * xb[idx],
            lam * yb + (1 - lam) * yb[idx],
            [lam * sv + (1 - lam) * sv[idx] for sv in sv_list_b],
            None if w_b is None else lam * w_b + (1 - lam) * w_b[idx])


# ─── FIXED: balanced loader uses combined/intersectional group ────────────────

def _make_loader(all_tensors, p, sv_tr_list, hard_idx, seed):
    """
    FIX #13: balance on worst-case subgroup considering ALL attrs via the intersectional.
    Uses intersectional group if available (last element of sv_tr_list when n_attrs > 1).
    """
    # Use intersectional group for balancing when available
    if len(sv_tr_list) > 1:
        sv_bal = sv_tr_list[-1]   # intersectional group (last appended)
    else:
        sv_bal = sv_tr_list[hard_idx]

    frac = sv_bal.mean()
    ds   = TensorDataset(*all_tensors)
    g    = torch.Generator()
    g.manual_seed(seed)

    if frac > 0.65 or frac < 0.35:
        idx0 = np.where(sv_bal == 0)[0]
        idx1 = np.where(sv_bal == 1)[0]
        rng2 = np.random.RandomState(seed)
        n_maj = max(len(idx0), len(idx1))
        idx0 = rng2.choice(idx0, n_maj, replace=True) if len(idx0) < n_maj else idx0
        idx1 = rng2.choice(idx1, n_maj, replace=True) if len(idx1) < n_maj else idx1
        idx  = np.concatenate([idx0, idx1])
        rng2.shuffle(idx)
        bal_ds = TensorDataset(*[t[idx] for t in all_tensors])
        return DataLoader(bal_ds, batch_size=p["batch"], shuffle=True,
                          drop_last=True, generator=g, num_workers=0)
    return DataLoader(ds, batch_size=p["batch"], shuffle=True,
                      drop_last=True, generator=g, num_workers=0)


# ─── Core Training ────────────────────────────────────────────────────────────

def _train_model(X_tr, y_tr, sv_tr_list,
                 X_val, y_val, sv_val_list,
                 X_test, y_test, sv_te_list,
                 params, ds_cfg, use_auditor, verbose, seed):

    p              = params
    n_base_attrs   = len(sv_tr_list)
    do_reweight    = ds_cfg.get("reweight_labels", False)
    grl_max        = p.get("grl_max", 0.5)
    fairness_coef  = p.get("fairness_coef", 8.0)
    mixup_a        = p.get("mixup_alpha", 0.0)
    task_lw        = p.get("task_lw", 1.0)
    min_pf         = p.get("min_pos_frac", 0.10)
    max_pf         = p.get("max_pos_frac", 0.90)
    min_acc        = p["min_acc_floor"]
    adv_lr_mult    = p.get("adv_lr_mult", 5.0)
    adv_steps      = p.get("adv_steps", 3)

    set_seed(seed)

    def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)

    Xt  = _t(X_tr);  yt  = _t(y_tr)
    Xv  = _t(X_val); Xte = _t(X_test)
    st_list = [_t(sv) for sv in sv_tr_list]

    wt = (torch.tensor(compute_label_reweights(y_tr, sv_tr_list),
                       dtype=torch.float32).to(DEVICE)
          if do_reweight
          else torch.ones(len(y_tr), dtype=torch.float32).to(DEVICE))

    hard_idx = find_hardest_attr(y_tr, sv_tr_list, mode="eo")

    in_dim  = X_tr.shape[1]
    rep_dim = p["hidden_dim"] // 2

    encoder   = Encoder(in_dim, p["hidden_dim"], p["dropout"]).to(DEVICE)
    task_head = TaskHead(rep_dim).to(DEVICE)
    sens_cls  = SensAttrClassifier(rep_dim, n_base_attrs,
                                   p["hidden_dim"], p["dropout"]).to(DEVICE)

    bce_task = nn.BCEWithLogitsLoss(reduction="none")
    bce_sens = nn.BCEWithLogitsLoss()

    # ── pre-training (task only) ──────────────────────────────────────────────
    opt_pre = optim.AdamW(
        list(encoder.parameters()) + list(task_head.parameters()),
        lr=p["lr"], weight_decay=1e-4)

    current_sv_tr_list = list(sv_tr_list)
    current_st_list    = list(st_list)
    n_adv_attrs        = n_base_attrs

    loader = _make_loader([Xt, yt, wt] + current_st_list, p,
                          current_sv_tr_list, hard_idx, seed)
    encoder.train(); task_head.train()
    for _ in range(p.get("pre_epochs", 40)):
        for batch in loader:
            xb = batch[0]; yb = batch[1]; wb = batch[2]
            sv_list_b = [batch[3 + i] for i in range(n_adv_attrs)]
            xb_m, yb_m, _, wb_m = mixup_batch(xb, yb, sv_list_b, alpha=mixup_a, w_b=wb)
            if wb_m is None: wb_m = wb
            opt_pre.zero_grad(set_to_none=True)
            ((bce_task(task_head(encoder(xb_m)), yb_m) * wb_m).mean()).backward()
            opt_pre.step()

    # ── FIX #6: Two optimisers — encoder/task + separate adversary ────────────
    def _make_opts(lr, adv_lr):
        opt_enc = optim.AdamW(
            list(encoder.parameters()) + list(task_head.parameters()),
            lr=lr, weight_decay=1e-4)
        opt_adv = optim.AdamW(
            list(sens_cls.parameters()),
            lr=adv_lr, weight_decay=1e-4)
        return opt_enc, opt_adv

    opt_enc, opt_adv = _make_opts(p["lr"], p["lr"] * adv_lr_mult)
    sched_enc = optim.lr_scheduler.CosineAnnealingLR(opt_enc, p["epochs"],
                                                      eta_min=p["lr"] * 0.01)

    # ── Training loop ─────────────────────────────────────────────────────────
    best_fair   = np.inf
    best_auc    = -np.inf
    best_enc    = copy.deepcopy(encoder.state_dict())
    best_th_sd  = copy.deepcopy(task_head.state_dict())
    no_imp      = 0
    auditor_log = []

    # Track val/test sv lists (updated when auditor fires)
    current_sv_val_list = list(sv_val_list)
    current_sv_te_list  = list(sv_te_list)

    pbar = tqdm(range(p["epochs"]),
                desc="  [AGAD]" if use_auditor else "  [AGAD-base]",
                dynamic_ncols=True, disable=not verbose)

    for epoch in pbar:
        grl_alpha = grl_max * float(epoch) / max(1, p["epochs"])
        sens_cls.set_alpha(grl_alpha)

        # ── Auditor step (every AUDIT_EVERY epochs, only if use_auditor) ──────
        if use_auditor and epoch > 0 and epoch % AUDIT_EVERY == 0:
            encoder.eval(); task_head.eval()

            # FIX #3: validate discovered subgroup on VAL set, not train
            with torch.no_grad():
                pv_val = torch.sigmoid(task_head(encoder(Xv))).cpu().numpy()

            # Auditor runs on val predictions vs val ground truth
            worst_eo, worst_grp, worst_mem_val = run_auditor(
                y_val, pv_val, [ensure_binary(sv) for sv in current_sv_val_list[:n_base_attrs]])

            if worst_grp is not None and worst_eo > TARGET_EO:
                # Map val membership back to train membership via same attributes
                # We re-discover the worst subgroup on training set using same definition
                active = worst_grp["active"]
                vals_req = worst_grp["vals"]
                n_tr = len(y_tr)
                tr_mem = np.ones(n_tr, dtype=bool)
                sv_tr_arr = [ensure_binary(sv) for sv in current_sv_tr_list[:n_base_attrs]]
                for ai, rv in zip(active, vals_req):
                    tr_mem &= (sv_tr_arr[ai] == rv)

                n_te = len(y_test)
                te_mem = np.ones(n_te, dtype=bool)
                sv_te_arr = [ensure_binary(sv) for sv in current_sv_te_list[:n_base_attrs]]
                for ai, rv in zip(active, vals_req):
                    te_mem &= (sv_te_arr[ai] == rv)

                new_sv_tr   = tr_mem.astype(int)
                new_sv_val  = worst_mem_val
                new_sv_te   = te_mem.astype(int)

                auditor_log.append({
                    "epoch": epoch, "worst_eo": worst_eo, "grp": worst_grp})
                tqdm.write(
                    f"  [auditor ep={epoch}] worst_val_EO={worst_eo:.4f}  grp={worst_grp}")

                # Only add new attr if we haven't already
                current_sv_tr_list  = list(sv_tr_list) + [new_sv_tr]
                current_sv_val_list = list(sv_val_list) + [new_sv_val]   # FIX #15
                current_sv_te_list  = list(sv_te_list)  + [new_sv_te]    # FIX #15
                current_st_list     = [_t(sv) for sv in current_sv_tr_list]
                n_adv_attrs         = len(current_sv_tr_list)
                sens_cls.set_n_attrs(n_adv_attrs)  # FIX #5: keeps old weights

                # FIX: don't reset lr to 0.1× every single audit; use a mild warmup
                opt_enc, opt_adv = _make_opts(p["lr"] * 0.5, p["lr"] * adv_lr_mult * 0.5)
                loader = _make_loader([Xt, yt, wt] + current_st_list,
                                      p, current_sv_tr_list, hard_idx, seed)

        # ── Batch training ────────────────────────────────────────────────────
        encoder.train(); task_head.train(); sens_cls.train()

        for batch in loader:
            xb = batch[0]; yb = batch[1]; wb = batch[2]
            sv_list_b = [batch[3 + i] for i in range(n_adv_attrs)]

            # FIX #6a: adversary update steps (sens_cls only)
            if grl_alpha > 0:
                for _ in range(adv_steps):
                    with torch.no_grad():
                        z_adv = encoder(xb).detach()
                    # adversary tries to PREDICT sensitive attrs from representation
                    adv_logits = sens_cls.out(sens_cls.hidden(z_adv))
                    adv_loss   = bce_sens(adv_logits,
                                         torch.stack(sv_list_b, dim=1))
                    opt_adv.zero_grad(set_to_none=True)
                    adv_loss.backward()
                    opt_adv.step()

            # FIX #6b: encoder + task update (tries to fool adversary via GRL)
            xb_m, yb_m, sv_m, wb_m = mixup_batch(xb, yb, sv_list_b,
                                                   alpha=mixup_a, w_b=wb)
            if wb_m is None: wb_m = wb

            opt_enc.zero_grad(set_to_none=True)
            z_m    = encoder(xb_m);  logits_m = task_head(z_m)
            z_raw  = encoder(xb);    logits_r = task_head(z_raw)
            task_loss = (bce_task(logits_m, yb_m) * wb_m).mean()

            # FIX #7: fairness penalty uses HARD thresholded predictions
            fair_pen = torch.tensor(0., device=DEVICE)
            with torch.no_grad():
                pred_hard = (torch.sigmoid(logits_r) > 0.5).float()
            for sv_b in sv_list_b:
                sf = sv_b.float(); yf = yb.float()
                prob = torch.sigmoid(logits_r)   # soft, for gradient
                pred = pred_hard                  # threshold for EO computation
                eps  = torch.tensor(1e-4, device=DEVICE)
                g0y1 = (1 - sf) * yf
                g1y1 = sf * yf
                g0y0 = (1 - sf) * (1 - yf)
                g1y0 = sf * (1 - yf)
                if any(g.sum() < 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
                    continue
                # Hard EO (non-differentiable; used as regulariser signal)
                tpr0_h = (pred * g0y1).sum() / (g0y1.sum() + eps)
                tpr1_h = (pred * g1y1).sum() / (g1y1.sum() + eps)
                fpr0_h = (pred * g0y0).sum() / (g0y0.sum() + eps)
                fpr1_h = (pred * g1y0).sum() / (g1y0.sum() + eps)
                eo_h   = torch.max(torch.abs(tpr0_h - tpr1_h),
                                   torch.abs(fpr0_h - fpr1_h))
                # Soft EO (differentiable, for gradient flow)
                tpr0 = (prob * g0y1).sum() / (g0y1.sum() + eps)
                tpr1 = (prob * g1y1).sum() / (g1y1.sum() + eps)
                fpr0 = (prob * g0y0).sum() / (g0y0.sum() + eps)
                fpr1 = (prob * g1y0).sum() / (g1y0.sum() + eps)
                eo_s  = torch.max(torch.abs(tpr0 - tpr1),
                                  torch.abs(fpr0 - fpr1))
                # Combine: use soft for gradient, weighted by hard magnitude
                fair_pen = fair_pen + eo_s * (1 + eo_h.detach())
            fair_pen = fair_pen / max(len(sv_list_b), 1)

            # FIX #4: GRL loss added once, NOT multiplied by alpha again
            # (GRL already reverses gradients scaled by alpha internally)
            grl_loss = (bce_sens(sens_cls(z_raw),
                                 torch.stack(sv_list_b, dim=1))
                        if grl_alpha > 0 else torch.tensor(0., device=DEVICE))

            pos_rate  = torch.sigmoid(logits_r).mean()
            rate_pen  = torch.tensor(0., device=DEVICE)
            if pos_rate < min_pf:
                d = torch.tensor(min_pf, device=DEVICE) - pos_rate
                rate_pen = d ** 2 * 100. + d * 10.
            elif pos_rate > max_pf:
                d = pos_rate - torch.tensor(max_pf, device=DEVICE)
                rate_pen = d ** 2 * 100. + d * 10.

            loss = (task_lw * task_loss
                    + fairness_coef * fair_pen
                    + grl_loss          # FIX #4: no extra alpha mult
                    + rate_pen)
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(task_head.parameters()), 0.5)
            opt_enc.step()

        sched_enc.step()

        # ── Validation / Early stopping ───────────────────────────────────────
        if epoch % p["eval_every"] == 0:
            encoder.eval(); task_head.eval()
            with torch.no_grad():
                pv_proba = torch.sigmoid(task_head(encoder(Xv))).cpu().numpy()
            val_pred = (pv_proba > 0.5).astype(int)
            val_acc  = accuracy_score(y_val, val_pred)
            val_pos  = val_pred.mean()
            val_auc  = safe_auc(y_val, pv_proba)
            valid    = (val_acc >= min_acc and min_pf <= val_pos <= max_pf)
            val_wc_sg, _, _ = compute_eo_subgroups(
                y_val, val_pred,
                [ensure_binary(sv) for sv in current_sv_val_list])

            # FIX #9: best = fairest model that is also within 1% AUC of best seen
            if val_auc > best_auc - 0.01 and val_auc > best_auc:
                best_auc = val_auc
            if valid and val_wc_sg < best_fair and val_auc >= best_auc - 0.01:
                best_fair   = val_wc_sg
                best_enc    = copy.deepcopy(encoder.state_dict())
                best_th_sd  = copy.deepcopy(task_head.state_dict())
                no_imp      = 0
            else:
                no_imp += p["eval_every"]

            if verbose:
                pbar.set_postfix({
                    "v_SG-EO": f"{val_wc_sg:.4f}",
                    "vacc":    f"{val_acc:.3f}",
                    "vauc":    f"{val_auc:.3f}",
                    "best":    f"{best_fair:.4f}",
                    "n_adv":   n_adv_attrs})

            if no_imp >= p["patience"] * p["eval_every"]:
                if verbose:
                    tqdm.write(f"\n  Early stop ep={epoch}")
                break

    # ── Evaluation on test set ────────────────────────────────────────────────
    encoder.load_state_dict(best_enc)
    task_head.load_state_dict(best_th_sd)
    encoder.eval(); task_head.eval()
    with torch.no_grad():
        proba_te = torch.sigmoid(task_head(encoder(Xte))).cpu().numpy()

    sv_te_np = [ensure_binary(sv) for sv in current_sv_te_list]
    pred     = (proba_te > 0.5).astype(int)

    per_eo, wc_eo = compute_eo_multi(y_test, pred, sv_te_np)
    per_dp, wc_dp = compute_dp_multi(pred, sv_te_np)
    sg_wc, sg_grp, _ = compute_eo_subgroups(y_test, pred, sv_te_np)
    ece = compute_ece(y_test, proba_te)
    acc = accuracy_score(y_test, pred)
    auc = safe_auc(y_test, proba_te)

    label  = "[agad]" if use_auditor else "[agad-base]"
    status = "PASS" if sg_wc < TARGET_EO else "FAIL"
    tqdm.write(
        f"  {label} pos={pred.mean():.2f}  WC-EO={wc_eo:.4f}  SG-EO={sg_wc:.4f}  "
        f"WC-DP={wc_dp:.4f}  acc={acc:.4f}  auc={auc:.4f}  "
        f"ECE={ece:.4f}  {status}")

    return {
        "eo": wc_eo, "sg_eo": sg_wc, "acc": acc, "auc": auc,
        "dp": wc_dp, "ece": ece,
        "per_attr_eo": per_eo, "per_attr_dp": per_dp,
        "subgroup_worst": sg_wc, "auditor_log": auditor_log,
    }


# ─── Logistic Regression Baseline ────────────────────────────────────────────

def run_lr_baseline(X_tr, y_tr, X_val, y_val, sv_val_list,
                    X_test, y_test, sv_te_list, params):
    clf = LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0, random_state=SEED)
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_test)[:, 1]
    pred  = (proba > 0.5).astype(int)
    sv_te_np = [ensure_binary(sv) for sv in sv_te_list]
    per_eo, wc_eo = compute_eo_multi(y_test, pred, sv_te_np)
    per_dp, wc_dp = compute_dp_multi(pred, sv_te_np)
    sg_w, _, _    = compute_eo_subgroups(y_test, pred, sv_te_np)
    ece           = compute_ece(y_test, proba)
    acc           = accuracy_score(y_test, pred)
    auc           = safe_auc(y_test, proba)
    status = "PASS" if sg_w < TARGET_EO else "FAIL"
    tqdm.write(
        f"  WC-EO={wc_eo:.4f}  SG-EO={sg_w:.4f}  WC-DP={wc_dp:.4f}  "
        f"per_attr_eo={[f'{v:.4f}' for v in per_eo]}  acc={acc:.4f}  {status}")
    return {
        "eo": wc_eo, "sg_eo": sg_w, "acc": acc, "auc": auc,
        "dp": wc_dp, "ece": ece,
        "per_attr_eo": per_eo, "per_attr_dp": per_dp,
        "subgroup_worst": sg_w, "auditor_log": [],
    }


# ─── Dataset runner ───────────────────────────────────────────────────────────

def _run_dataset(data, ds_key):
    cfg        = DATASET_CONFIG[ds_key]
    p          = _get_params(ds_key)
    attr_names = cfg["sens_attrs"]

    X_train = to_dense(data["X_train_nn"])
    X_test  = to_dense(data["X_test_nn"])
    y_train = np.array(data["y_train"])
    y_test  = np.array(data["y_test"])

    sv_tr_full = [ensure_binary(np.array(data[f"{a}_train"])) for a in attr_names]
    sv_te      = [ensure_binary(np.array(data[f"{a}_test"]))  for a in attr_names]

    if len(attr_names) > 1:
        # FIX #8: true intersection (minority × minority), not majority-vs-rest
        sv_tr_full.append(make_intersectional(sv_tr_full))
        sv_te.append(make_intersectional(sv_te))

    results = {m: {k: [] for k in
                   ["eo", "sg_eo", "acc", "auc", "dp", "ece",
                    "per_attr_eo", "per_attr_dp", "subgroup_worst", "auditor_log"]}
               for m in METHODS}

    for seed in SEEDS:
        tqdm.write(f"\n  ── [AGAD] seed {seed} {'[FAST]' if FAST_MODE else ''} ──")
        tr_idx, val_idx = build_val_split(y_train, frac=0.15, seed=seed)
        X_tr,  X_val   = X_train[tr_idx], X_train[val_idx]
        y_tr,  y_val   = y_train[tr_idx], y_train[val_idx]
        sv_tr  = [sv[tr_idx]  for sv in sv_tr_full]
        sv_val = [sv[val_idx] for sv in sv_tr_full]

        tqdm.write("  [lr_baseline]")
        r = run_lr_baseline(X_tr, y_tr, X_val, y_val, sv_val,
                            X_test, y_test, sv_te, p)
        for k in results["lr_baseline"]:
            results["lr_baseline"][k].append(r[k])

        tqdm.write("  [agad_no_auditor]")
        r_base = _train_model(X_tr, y_tr, sv_tr, X_val, y_val, sv_val,
                              X_test, y_test, sv_te,
                              p, cfg, use_auditor=False, verbose=True, seed=seed)
        for k in results["agad_no_auditor"]:
            results["agad_no_auditor"][k].append(r_base[k])

        tqdm.write("  [agad] (auditor-guided)")
        r_agad = _train_model(X_tr, y_tr, sv_tr, X_val, y_val, sv_val,
                              X_test, y_test, sv_te,
                              p, cfg, use_auditor=True, verbose=True, seed=seed)
        for k in results["agad"]:
            results["agad"][k].append(r_agad[k])

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


# ─── Preprocessing ────────────────────────────────────────────────────────────

def _cache_path(name):
    return f"/tmp/agad_{CACHE_VERSION}_{name}.pkl"

def preprocess_adult(csv_path, use_cache=True):
    cache = _cache_path("adult")
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip().replace(".", "_") for c in df.columns]
    for c in df.select_dtypes("object").columns:
        df[c] = df[c].str.strip()
    df["income_binary"] = (~df["income"].isin(["<=50K", "<=50K."])).astype(int)
    df["sex_binary"]    = (df["sex"]  == "Male").astype(int)
    df["race_binary"]   = (df["race"] == "White").astype(int)
    num_cols = ["age", "fnlwgt", "education_num", "capital_gain",
                "capital_loss", "hours_per_week"]
    cat_cols = ["workclass", "education", "marital_status",
                "occupation", "relationship", "native_country"]
    for c in num_cols:
        df[c] = clean_numeric(df[c])
    df["log_capital_gain"] = np.log1p(df["capital_gain"].clip(lower=0))
    df["log_capital_loss"] = np.log1p(df["capital_loss"].clip(lower=0))
    df["age_hours"]        = df["age"] * df["hours_per_week"]
    num_cols = num_cols + ["log_capital_gain", "log_capital_loss", "age_hours"]
    drop_cols = {"income", "income_binary", "sex_binary", "race_binary", "race", "sex"}
    X  = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y  = df["income_binary"].values
    ss = df["sex_binary"].values
    sr = df["race_binary"].values
    tr, te = stratified_split(X.values, y, [ss, sr])
    preproc = ColumnTransformer([
        ("num", StandardScaler(),                                         num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
    # FIX #1 (adult already splits first - correct here, kept same)
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_sex_train": ss[tr], "sens_sex_test": ss[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
    }
    joblib.dump(res, cache)
    return res

def preprocess_compas(csv_path, use_cache=True):
    cache = _cache_path("compas")
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df["days_b_screening_arrest"].between(-30, 30)) &
        (df["is_recid"] != -1) &
        (df["c_charge_degree"] != "O") &
        (df["score_text"] != "N/A"),
        ["age", "c_charge_degree", "race", "age_cat", "sex", "priors_count",
         "days_b_screening_arrest", "juv_other_count", "juv_misd_count",
         "juv_fel_count", "c_charge_desc", "is_recid", "two_year_recid",
         "c_jail_in", "c_jail_out"]
    ].reset_index(drop=True)
    df["race_binary"] = (df["race"] == "African-American").astype(int)
    df["sex_binary"]  = (df["sex"]  == "Female").astype(int)
    df["c_jail_in"]  = pd.to_datetime(df["c_jail_in"],  errors="coerce")
    df["c_jail_out"] = pd.to_datetime(df["c_jail_out"], errors="coerce")
    df["jail_time"]  = (df["c_jail_out"] - df["c_jail_in"]).dt.days.fillna(0).clip(lower=0)
    df = df.drop(columns=["c_jail_in", "c_jail_out"])
    df["log_priors"] = np.log1p(df["priors_count"])
    df["young_adult"] = (df["age"] < 25).astype(int)
    num_cols = ["age", "priors_count", "days_b_screening_arrest", "jail_time",
                "juv_other_count", "juv_misd_count", "juv_fel_count",
                "log_priors", "young_adult"]
    cat_cols = ["c_charge_degree", "race", "age_cat", "sex", "c_charge_desc"]
    for c in num_cols:
        df[c] = clean_numeric(df[c])
    preproc = ColumnTransformer([
        ("num", StandardScaler(),                                         num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
    X  = df.drop(columns=["is_recid", "two_year_recid", "race_binary", "sex_binary"])
    y  = df["two_year_recid"].values
    sr = df["race_binary"].values
    ss = df["sex_binary"].values
    tr, te = stratified_split(X.values, y, [sr, ss])
    res = {
        "X_train_nn": preproc.fit_transform(X.iloc[tr].reset_index(drop=True)),
        "X_test_nn":  preproc.transform(X.iloc[te].reset_index(drop=True)),
        "y_train": y[tr], "y_test": y[te],
        "sens_race_train": sr[tr], "sens_race_test": sr[te],
        "sens_sex_train":  ss[tr], "sens_sex_test":  ss[te],
    }
    joblib.dump(res, cache)
    return res

def preprocess_acs_employment(use_cache=True):
    """FIX #1: scaler fitted on train only."""
    cache = _cache_path("acs_employment")
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    if not HAS_FOLKTABLES:
        raise RuntimeError("folktables not installed: pip install folktables")
    data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs_data    = data_source.get_data(states=["CA"], download=True)
    features, label, _group = ACSEmployment.df_to_numpy(acs_data)
    acs_eligible = acs_data.iloc[:len(label)]
    race_binary = (acs_eligible["RAC1P"].values != 1).astype(int)
    sex_binary  = (acs_eligible["SEX"].values   == 2).astype(int)
    valid = ~np.isnan(label)
    features    = features[valid].astype(np.float32)
    label       = label[valid].astype(int)
    race_binary = race_binary[valid]
    sex_binary  = sex_binary[valid]
    # FIX: split FIRST, scale on train only
    tr, te = stratified_split(features, label, [race_binary, sex_binary])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(features[tr])
    X_te_s = scaler.transform(features[te])
    res = {
        "X_train_nn": X_tr_s, "X_test_nn": X_te_s,
        "y_train": label[tr], "y_test": label[te],
        "sens_race_train": race_binary[tr], "sens_race_test": race_binary[te],
        "sens_sex_train":  sex_binary[tr],  "sens_sex_test":  sex_binary[te],
    }
    joblib.dump(res, cache)
    return res

def preprocess_acs_public_coverage(use_cache=True):
    """FIX #1: scaler fitted on train only."""
    cache = _cache_path("acs_public_coverage")
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    if not HAS_FOLKTABLES:
        raise RuntimeError("folktables not installed: pip install folktables")
    data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs_data    = data_source.get_data(states=["CA"], download=True)
    features, label, _group = ACSPublicCoverage.df_to_numpy(acs_data)
    acs_eligible = acs_data.iloc[:len(label)]
    race_binary = (acs_eligible["RAC1P"].values != 1).astype(int)
    sex_binary  = (acs_eligible["SEX"].values   == 2).astype(int)
    valid = ~np.isnan(label)
    features    = features[valid].astype(np.float32)
    label       = label[valid].astype(int)
    race_binary = race_binary[valid]
    sex_binary  = sex_binary[valid]
    tr, te = stratified_split(features, label, [race_binary, sex_binary])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(features[tr])
    X_te_s = scaler.transform(features[te])
    res = {
        "X_train_nn": X_tr_s, "X_test_nn": X_te_s,
        "y_train": label[tr], "y_test": label[te],
        "sens_race_train": race_binary[tr], "sens_race_test": race_binary[te],
        "sens_sex_train":  sex_binary[tr],  "sens_sex_test":  sex_binary[te],
    }
    joblib.dump(res, cache)
    return res

PREPROCESS_FNS = {
    "adult":               lambda: preprocess_adult(DATA_PATHS["adult"]),
    "compas":              lambda: preprocess_compas(DATA_PATHS["compas"]),
    "acs_employment":      lambda: preprocess_acs_employment(),
    "acs_public_coverage": lambda: preprocess_acs_public_coverage(),
}


# ─── Results Reporting ────────────────────────────────────────────────────────

def print_results(all_results):
    W = 120
    target_eo = TARGET_EO
    target_dp = TARGET_DP
    fast_tag  = "  [FAST MODE — indicative only]" if FAST_MODE else ""
    seeds_tag = f"  seeds={SEEDS}  N={N_SEEDS}"
    print(f"\n{'═' * W}")
    print(f"  AGAD RESULTS   primary: SG-EO < {target_eo}   DP < {target_dp}{fast_tag}")
    print(f"  {seeds_tag}")
    print(f"  Research Q: Auditor-guided adversarial debiasing vs fairness gerrymandering")
    method_labels = {
        "lr_baseline":       "LR baseline",
        "agad_no_auditor":   "AGAD (no auditor)",
        "agad":              "AGAD (ours)",
    }
    for ds_key, mr in all_results.items():
        cfg = DATASET_CONFIG[ds_key]
        print(f"\n  ── {ds_key.upper()} (attrs: {'+'.join(cfg['sens_attrs'])}) ──")
        print(f"  {'Method':<22} {'WC-EO':>8} {'SG-EO':>8} {'WC-DP':>8} "
              f"{'Acc':>7} {'AUC':>7} {'ECE':>6}  Status")
        print(f"  {'─' * W}")
        for m in METHODS:
            if m not in mr or not mr[m]["sg_eo"]:
                continue
            def _ci(key):
                vals = [v for v in mr[m][key] if not (isinstance(v, float) and np.isnan(v))]
                if not vals: return float("nan"), float("nan"), float("nan")
                if len(vals) == 1: return vals[0], vals[0], vals[0]
                return bootstrap_ci(vals)
            eo_m, eo_lo, eo_hi = _ci("eo")
            sg_m, sg_lo, sg_hi = _ci("sg_eo")
            dp_m, dp_lo, dp_hi = _ci("dp")
            ac_m, _,     _     = _ci("acc")
            au_m, _,     _     = _ci("auc")
            ec_m, _,     _     = _ci("ece")
            status_eo = "PASS" if sg_m < target_eo else "FAIL"
            status_dp = "PASS" if dp_m < target_dp else "FAIL"
            status    = f"EO:{status_eo}  DP:{status_dp}"
            label     = method_labels.get(m, m)
            print(f"  {label:<22} {eo_m:>8.4f} {sg_m:>8.4f} {dp_m:>8.4f} "
                  f"{ac_m:>7.4f} {au_m:>7.4f} {ec_m:>6.4f}  {status}")
            if N_SEEDS > 1:
                print(f"  {'':22} [{sg_lo:.4f},{sg_hi:.4f}] 95%CI SG-EO"
                      f"  [{dp_lo:.4f},{dp_hi:.4f}] 95%CI DP")

        if "agad" in mr and mr["agad"]["auditor_log"]:
            n_audits = len(mr["agad"]["auditor_log"][0]) if mr["agad"]["auditor_log"] else 0
            print(f"  auditor fired ~{n_audits} time(s) per seed")

        if "agad" in mr and "agad_no_auditor" in mr:
            sg_base = np.nanmean(mr["agad_no_auditor"]["sg_eo"])
            sg_agad = np.nanmean(mr["agad"]["sg_eo"])
            improved = "improved" if sg_agad < sg_base else "no gain"
            print(f"  auditor gain: SG-EO {sg_base:.4f} → {sg_agad:.4f}  "
                  f"delta={sg_base - sg_agad:.4f} ({improved})")

    print(f"\n{'═' * W}")
    full_pass = sum(
        1 for mr in all_results.values()
        if "agad" in mr and mr["agad"]["sg_eo"]
        and np.nanmean(mr["agad"]["sg_eo"]) < target_eo)
    print(f"  AGAD EO-PASS: {full_pass}/{len(all_results)} datasets")
    dp_pass = sum(
        1 for mr in all_results.values()
        if "agad" in mr and mr["agad"]["dp"]
        and np.nanmean(mr["agad"]["dp"]) < target_dp)
    print(f"  AGAD DP-PASS: {dp_pass}/{len(all_results)} datasets")
    if FAST_MODE:
        print("  NOTE: fast mode. Rerun with FAST_MODE=False for final results.")
    if N_SEEDS > 1:
        print(f"  NOTE: results are mean ± 95% bootstrap CI over {N_SEEDS} seeds.")
    else:
        print("  NOTE: single seed. Set N_SEEDS≥3 for confidence intervals.")

    print("""
  ─── Notes on EO=0.05 threshold ───────────────────────────────────────────────
  The 0.05 EO target used here is consistent with the IBM AIF360 "acceptable bias"
  convention (avg odds diff ±0.05) and frequently cited in ML fairness literature.
  It is a research benchmark, not a legal threshold. COMPAS naturally has higher
  baseline EO (~0.29) due to correlated base rates; reaching <0.05 there requires
  either heavy fairness pressure (accuracy cost) or post-processing. Reporting
  both EO and DP is recommended: DP captures outcome-rate parity (easier to meet),
  EO captures error-rate parity (more meaningful for high-stakes decisions).
  """)


def save_results(all_results, path="/kaggle/working/results_agad.json"):
    def _clean(obj):
        if isinstance(obj, dict):  return {k: _clean(v) for k, v in obj.items()}
        if isinstance(obj, list):  return [_clean(v) for v in obj]
        if isinstance(obj, float) and np.isnan(obj): return None
        if isinstance(obj, np.floating):  return float(obj)
        if isinstance(obj, np.integer):   return int(obj)
        if isinstance(obj, np.ndarray):   return obj.tolist()
        if isinstance(obj, torch.Tensor): return obj.tolist()
        return obj
    suffix   = "_fast" if FAST_MODE else ""
    out_path = path.replace(".json", f"{suffix}.json")
    with open(out_path, "w") as f:
        json.dump(_clean({"results": all_results, "seeds": SEEDS,
                          "cache_version": CACHE_VERSION}), f, indent=2)
    tqdm.write(f"\n  Results saved → {out_path}")


# ─── Main ─────────────────────────────────────────────────────────────────────

def main():
    all_results = {}
    t0 = time.time()
    if FAST_MODE:
        tqdm.write("[FAST MODE] epochs reduced.")
    for ds_key in RUN_DATASETS:
        tqdm.write(f"\n{'#' * 78}")
        tqdm.write(f"  DATASET: {ds_key.upper()}")
        tqdm.write(f"{'#' * 78}")
        data = PREPROCESS_FNS[ds_key]()
        y_te = np.array(data["y_test"])
        cfg  = DATASET_CONFIG[ds_key]
        sv_te_p = ensure_binary(np.array(data[f"{cfg['sens_attrs'][0]}_test"]))
        tqdm.write(
            f"\n  n_train={len(data['y_train'])}  n_test={len(y_te)}"
            f"  label_pos={y_te.mean():.2f}  sens_pos={sv_te_p.mean():.2f}"
            f"  attrs={cfg['sens_attrs']}")
        all_results[ds_key] = _run_dataset(data, ds_key)
        del data; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print_results(all_results)
    save_results(all_results)
    print(f"\n  Total wall time: {(time.time() - t0) / 60:.1f} min")
    return all_results

if __name__ == "__main__":
    main()


##############################################################################
  DATASET: ADULT
##############################################################################

  n_train=26048  n_test=6513  label_pos=0.24  sens_pos=0.67  attrs=['sens_sex', 'sens_race']

  ── [AGAD] seed 42  ──
  [lr_baseline]
  WC-EO=0.2099  SG-EO=0.2099  WC-DP=0.1850  per_attr_eo=['0.0947', '0.0375', '0.2099']  acc=0.8511  FAIL
  [agad_no_auditor]


  [AGAD-base]:   0%|          | 0/150 [00:00<?, ?it/s]

  [agad-base] pos=0.19  WC-EO=0.1198  SG-EO=0.1198  WC-DP=0.1248  acc=0.8465  auc=0.8867  ECE=0.0405  FAIL
  [agad] (auditor-guided)


  [AGAD]:   0%|          | 0/150 [00:00<?, ?it/s]

  [auditor ep=5] worst_val_EO=0.1940  grp={'active': [0, 1], 'vals': [0, 0], 'size': 269}
  [auditor ep=10] worst_val_EO=0.1269  grp={'active': [0, 1], 'vals': [0, 0], 'size': 269}
  [auditor ep=15] worst_val_EO=0.1290  grp={'active': [0, 1], 'vals': [0, 0], 'size': 269}
  [auditor ep=20] worst_val_EO=0.0838  grp={'active': [0, 1], 'vals': [0, 1], 'size': 1010}
  [auditor ep=25] worst_val_EO=0.0576  grp={'active': [0, 1], 'vals': [0, 0], 'size': 269}
  [auditor ep=30] worst_val_EO=0.0904  grp={'active': [0, 1], 'vals': [1, 0], 'size': 320}
  [auditor ep=35] worst_val_EO=0.1016  grp={'active': [0, 1], 'vals': [0, 1], 'size': 1010}
  [auditor ep=40] worst_val_EO=0.0988  grp={'active': [0, 1], 'vals': [0, 1], 'size': 1010}
  [auditor ep=45] worst_val_EO=0.0950  grp={'active': [0, 1], 'vals': [1, 0], 'size': 320}
  [auditor ep=50] worst_val_EO=0.1563  grp={'active': [0, 1], 'vals': [0, 1], 'size': 1010}
  [auditor ep=55] worst_val_EO=0.1258  grp={'active': [0, 1], 'vals': [0, 1], 'size': 1

...

  [auditor ep=15] worst_val_EO=0.0627  grp={'active': [0, 1], 'vals': [1, 0], 'size': 3252}
  [auditor ep=20] worst_val_EO=0.0502  grp={'active': [0, 1], 'vals': [1, 0], 'size': 3252}
  [auditor ep=70] worst_val_EO=0.0502  grp={'active': [0, 1], 'vals': [1, 0], 'size': 3252}
  [agad] pos=0.23  WC-EO=0.0088  SG-EO=0.0088  WC-DP=0.0045  acc=0.7193  auc=0.7582  ECE=0.0155  PASS

  ── [AGAD] seed 44  ──
  [lr_baseline]
  WC-EO=0.0123  SG-EO=0.0215  WC-DP=0.0057  per_attr_eo=['0.0123', '0.0120', '0.0098']  acc=0.6874  PASS
  [agad_no_auditor]


  [AGAD-base]:   0%|          | 0/120 [00:00<?, ?it/s]

  [agad-base] pos=0.24  WC-EO=0.0149  SG-EO=0.0149  WC-DP=0.0064  acc=0.7216  auc=0.7587  ECE=0.0150  PASS
  [agad] (auditor-guided)


  [AGAD]:   0%|          | 0/120 [00:00<?, ?it/s]

  [auditor ep=90] worst_val_EO=0.0506  grp={'active': [0, 1], 'vals': [1, 0], 'size': 3290}
  [agad] pos=0.24  WC-EO=0.0149  SG-EO=0.0149  WC-DP=0.0069  acc=0.7216  auc=0.7587  ECE=0.0150  PASS

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  AGAD RESULTS   primary: SG-EO < 0.05   DP < 0.1
    seeds=[42, 43, 44]  N=3
  Research Q: Auditor-guided adversarial debiasing vs fairness gerrymandering

  ── ADULT (attrs: sens_sex+sens_race) ──
  Method                    WC-EO    SG-EO    WC-DP     Acc     AUC    ECE  Status
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  LR baseline              0.2127   0.2127   0.1849  0.8518  0.9049 0.0096  EO:FAIL  DP:FAIL
                         [0.2099,0.2151] 95%CI SG-EO  [0.1848,0.1850] 95%CI DP
  AGAD (no auditor)        0.0718   0.0733   0.1360  0.8453  0.8889 0.0414  EO:FAIL  DP:FAIL
              